# Lesson 3: Classes

Enough object-oriented programming to read and write a coffea analysis.

<img src="img/map-3.svg" width="85%" alt="">

## Objects

Everything in Python is an object: a value that carries data and methods.

A method is a function attached to a value. You call it with a dot.

In [ ]:
import numpy as np

pt = np.array([63.0, 38.1, 4.0])
pt.max(), pt.shape, "muon".upper()

## Classes

A class is a recipe. Objects made from it each have their own data.

<img src="img/class-objects.svg" width="85%" alt="">

## `__init__` and `self`

`__init__` sets up a new object. `self` is the object being built.

In [ ]:
class Muon:
    def __init__(self, pt, charge):
        self.pt = pt
        self.charge = charge

mu = Muon(63.0, 1)
mu.pt

## Separate data

Each object keeps its own data.

In [ ]:
other = Muon(38.1, -1)
mu.pt, other.pt

## Methods

A method is a function defined inside the class. Its first argument is always `self`.

In [ ]:
class Muon:
    def __init__(self, pt, charge):
        self.pt = pt
        self.charge = charge

    def opposite(self, other):
        return self.charge != other.charge

Muon(63.0, 1).opposite(Muon(38.1, -1))

## Properties

`@property` makes a method read like data: `z.mass`, with no brackets.

This is how `vector` gives you `.mass` in lesson 4.

In [ ]:
class FourVector:
    def __init__(self, E, px, py, pz):
        self.E = E
        self.px = px
        self.py = py
        self.pz = pz

    @property
    def mass(self):
        return np.sqrt(self.E**2 - self.px**2 - self.py**2 - self.pz**2)

FourVector(5.0, 1.0, 2.0, 2.0).mass

## `__add__`

`__add__` tells Python what `+` means for your objects.

Names with two underscores on each side are special methods. Python calls them for you.

In [ ]:
class FourVector:
    def __init__(self, E, px, py, pz):
        self.E = E
        self.px = px
        self.py = py
        self.pz = pz

    def __add__(self, other):
        return FourVector(
            self.E + other.E,
            self.px + other.px,
            self.py + other.py,
            self.pz + other.pz,
        )

    @property
    def mass(self):
        return np.sqrt(self.E**2 - self.px**2 - self.py**2 - self.pz**2)

(FourVector(5.0, 1.0, 2.0, 2.0) + FourVector(5.0, -1.0, -2.0, -2.0)).mass

## Columns of particles

The same class works for one particle or for a whole column of particles.

If the data are NumPy arrays, every method works element by element.

In [ ]:
E = np.array([5.0, 10.0])
px = np.array([1.0, 6.0])
zeros = np.zeros(2)
FourVector(E, px, zeros, zeros).mass

## Settings

A class can hold settings that its methods use.

In [ ]:
class Selection:
    def __init__(self, min_pt):
        self.min_pt = min_pt

    def apply(self, pt):
        return pt[pt > self.min_pt]

Selection(min_pt=20).apply(pt)

## Inheritance

A new class starts from an existing one and changes only what differs.

`super().__init__(...)` runs the parent's `__init__`.

In [ ]:
class TightSelection(Selection):
    def __init__(self):
        super().__init__(min_pt=50)

TightSelection().apply(pt)

## coffea processors

In coffea, an analysis is a class with a `process(self, events)` method.

You write the method. coffea splits the data into chunks, calls `process` on each chunk and adds up the outputs.

<img src="img/process-chunks.svg" width="85%" alt="">

## A processor in plain Python

Settings go in `__init__`, the analysis in `process`.

In [ ]:
class CountProcessor:
    def __init__(self, min_pt):
        self.min_pt = min_pt

    def process(self, events):
        return {"events": len(events), "passed": int((events > self.min_pt).sum())}

processor = CountProcessor(min_pt=20)
processor.process(np.array([63.0, 38.1, 4.0]))

## Adding outputs

Outputs from separate chunks add up, key by key.

In [ ]:
chunks = [np.array([63.0, 38.1, 4.0]), np.array([54.3, 23.5]), np.array([12.0])]
total = processor.process(chunks[0])
for chunk in chunks[1:]:
    output = processor.process(chunk)
    total = {key: total[key] + output[key] for key in total}
total

## What you can now read

| You see | It means |
|---|---|
| `class Muon:` | define a recipe for objects |
| `def __init__(self, ...):` | set up a new object |
| `self.pt = pt` | store data on the object |
| `mu.pt`, `mu.opposite(other)` | read data, call a method |
| `@property` | a method you read without brackets |
| `__add__` | what `+` means for your objects |
| `class B(A):`, `super()` | start from an existing class |

## Now: project 3.

[3b-project-classes.ipynb](3b-project-classes.ipynb): a four-vector class and a Higgs processor.

<img src="img/class-objects.svg" width="70%" alt="">